# Anime Recommendation - Modèles Avancés (Yanis)

## Objectifs :
- Tester XGBoost
- Tester LightGBM
- Tester MLP (réseau de neurones)
- Comparer les performances
- Optimiser le meilleur modèle

In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

# Chargement du dataset

In [67]:
df = pd.read_csv(
    "/kaggle/input/datasets/yanishelali/anime-ty-clean/anime_tv_clean.csv"
)

df.head()

,anime_id,episodes,rating,members,Action,Adventure,Cars,Comedy,Dementia,Demons,...,Shoujo Ai,Shounen,Shounen Ai,Slice of Life,Space,Sports,Super Power,Supernatural,Thriller,Vampire
0,5114,64.0,9.26,793665,1,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
1,28977,51.0,9.25,114262,1,0,0,1,0,0,...,0,1,0,0,0,0,0,0,0,0
2,9253,24.0,9.17,673572,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,9969,51.0,9.16,151266,1,0,0,1,0,0,...,0,1,0,0,0,0,0,0,0,0
4,32935,10.0,9.15,93351,0,0,0,1,0,0,...,0,1,0,0,0,1,0,0,0,0


# Informations sur le dataset

In [68]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3777 entries, 0 to 3776
Data columns (total 44 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   anime_id       3777 non-null   int64  
 1   episodes       3777 non-null   float64
 2   rating         3777 non-null   float64
 3   members        3777 non-null   int64  
 4   Action         3777 non-null   int64  
 5   Adventure      3777 non-null   int64  
 6   Cars           3777 non-null   int64  
 7   Comedy         3777 non-null   int64  
 8   Dementia       3777 non-null   int64  
 9   Demons         3777 non-null   int64  
 10  Drama          3777 non-null   int64  
 11  Ecchi          3777 non-null   int64  
 12  Fantasy        3777 non-null   int64  
 13  Game           3777 non-null   int64  
 14  Harem          3777 non-null   int64  
 15  Historical     3777 non-null   int64  
 16  Horror         3777 non-null   int64  
 17  Josei          3777 non-null   int64  
 18  Kids    

# Nettoyage des données

In [69]:
df = pd.read_csv(
    "/kaggle/input/datasets/yanishelali/anime-ty-clean/anime_tv_clean.csv"
)

df.columns = df.columns.str.replace(" ", "_")

df.head()

,anime_id,episodes,rating,members,Action,Adventure,Cars,Comedy,Dementia,Demons,...,Shoujo_Ai,Shounen,Shounen_Ai,Slice_of_Life,Space,Sports,Super_Power,Supernatural,Thriller,Vampire
0,5114,64.0,9.26,793665,1,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
1,28977,51.0,9.25,114262,1,0,0,1,0,0,...,0,1,0,0,0,0,0,0,0,0
2,9253,24.0,9.17,673572,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,9969,51.0,9.16,151266,1,0,0,1,0,0,...,0,1,0,0,0,0,0,0,0,0
4,32935,10.0,9.15,93351,0,0,0,1,0,0,...,0,1,0,0,0,1,0,0,0,0


In [70]:
import pandas as pd

# Chargement dataset clean
df = pd.read_csv(
    "/kaggle/input/datasets/yanishelali/anime-ty-clean/anime_tv_clean.csv"
)

# Fix LightGBM warning (optionnel)
df.columns = df.columns.str.replace(" ", "_")

# Vérification
print(df.columns)

# ML direct
X = df.drop("rating", axis=1)
y = df["rating"]

Index(['anime_id', 'episodes', 'rating', 'members', 'Action', 'Adventure',
       'Cars', 'Comedy', 'Dementia', 'Demons', 'Drama', 'Ecchi', 'Fantasy',
       'Game', 'Harem', 'Historical', 'Horror', 'Josei', 'Kids', 'Magic',
       'Martial_Arts', 'Mecha', 'Military', 'Music', 'Mystery', 'Parody',
       'Police', 'Psychological', 'Romance', 'Samurai', 'School', 'Sci-Fi',
       'Seinen', 'Shoujo', 'Shoujo_Ai', 'Shounen', 'Shounen_Ai',
       'Slice_of_Life', 'Space', 'Sports', 'Super_Power', 'Supernatural',
       'Thriller', 'Vampire'],
      dtype='object')


# Séparation des variables explicatives et de la variable cible

In [71]:
X = df.drop("rating", axis=1)
y = df["rating"]

# Division Train/Test

In [72]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fonction d'évaluation des modèles

In [73]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_model(name, model, X_test, y_test):
    pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    print(f"\n{name}")
    print(f"RMSE : {rmse:.3f}")
    print(f"MAE  : {mae:.3f}")
    print(f"R²   : {r2:.3f}")

    return rmse, mae, r2

# XGBoost

In [74]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    random_state=42
)

xgb.fit(X_train, y_train)

xgb_rmse, xgb_mae, xgb_r2 = evaluate_model(
    "XGBoost",
    xgb,   
    X_test,
    y_test
)


XGBoost
RMSE : 0.580
MAE  : 0.426
R²   : 0.559


In [75]:
print("Résultats XGBoost")

xgb_rmse, xgb_mae, xgb_r2 = evaluate_model(
    "XGBoost",
    xgb,
    X_test,
    y_test
)

Résultats XGBoost

XGBoost
RMSE : 0.580
MAE  : 0.426
R²   : 0.559


## Modèle 2 : LightGBM

In [76]:
!pip install lightgbm -q

from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    random_state=42
)

lgbm.fit(X_train, y_train)

print("Résultats LightGBM")

lgbm_rmse, lgbm_mae, lgbm_r2 = evaluate_model(
    "LightGBM",
    lgbm,
    X_test,
    y_test
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003396 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 699
[LightGBM] [Info] Number of data points in the train set: 3021, number of used features: 41
[LightGBM] [Info] Start training from score 6.897537
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

## Modèle 3 : Réseau de neurones (MLP)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    max_iter=300,
    random_state=42
)

mlp.fit(X_train_scaled, y_train)

print("Résultats MLP")

mlp_rmse, mlp_mae, mlp_r2 = evaluate_model(
    "MLP",
    mlp,
    X_test_scaled,
    y_test
)
